# RL Agent Deployment Tutorial

**Backtesting and deploying Q-Learning / RL agents**

This tutorial shows how to:

1. **Backtest an agent** with `BacktestRunner` and a simple environment
2. **Use the orchestrator pipeline** `rl.backtest_agent` (with stub or saved agent)
3. **Load a saved agent** for deployment
4. **Run the deploy pipeline** `rl.deploy_agent`

**References:** `docs/reference/q_learning/rl_framework.md`, `environments.md`, `runners.md`, `docs/guides/q_learning/deploying_rl_agents.md`

---

## 1. Setup and imports

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
np.random.seed(42)
print("Path and seed set.")

## 2. Backtest with BacktestRunner (standalone)

Use a **minimal environment** (`BaseEnv`) and a **simple agent** that implements `RLAgent`. Then run `BacktestRunner`.

In [ ]:
from src.q_learning.environments.base import BaseEnv
from src.q_learning.runners.backtest import BacktestRunner, BacktestConfig

# Minimal agent: random actions (for demo only)
class RandomAgent:
    def __init__(self, n_actions=3, seed=42):
        self._rng = np.random.default_rng(seed)
        self.n_actions = n_actions
    def select_action(self, state, *, training=False, explore=True):
        return self._rng.integers(0, self.n_actions)
    def update(self, transitions=None, batch=None):
        return None
    def get_parameters(self):
        return {}
    def set_parameters(self, params):
        pass

env = BaseEnv(state_dim=1, n_actions=3, max_steps=50, seed=42)
agent = RandomAgent(n_actions=3)
config = BacktestConfig(n_episodes=10, compute_sharpe=True)
runner = BacktestRunner(agent=agent, env=env, config=config)
result = runner.run()

print(f"Episodes: {result.n_episodes}")
print(f"Mean return: {result.mean_pnl_return:.4f}")
print(f"Sharpe: {result.sharpe_ratio:.2f}")
print(f"Max drawdown: {result.max_drawdown:.4f}")

## 3. Run the rl.backtest_agent pipeline

The orchestrator pipeline builds a **stub agent** and **BaseEnv** from config when no saved agent is provided, then runs the backtest and stores the result in context state.

In [ ]:
from pathlib import Path
from src.orchestrator.pipelines.rl.backtest_agent import build_pipeline
from src.orchestrator.core.pipeline import PipelineRunner
from src.orchestrator.core.context import Context
from src.orchestrator.config.schemas import RunConfig
from src.orchestrator.artifacts.store import ArtifactStore
from src.orchestrator.core.state_keys import StateKeys as Keys

config = RunConfig(
    pipeline="rl.backtest_agent",
    params={
        "rl": {
            "backtest": {
                "n_episodes": 15,
                "state_dim": 1,
                "n_actions": 3,
                "max_steps": 40,
                "seed": 123,
                "compute_sharpe": True,
                "compute_drawdown": True,
            },
        },
    },
)

artifacts_root = Path("./artifacts/rl_tutorial")
artifacts_root.mkdir(parents=True, exist_ok=True)
ctx = Context(
    run_id="rl-tutorial-1",
    cfg=config,
    logger=None,
    artifact_store=ArtifactStore(artifacts_root=artifacts_root),
)
pipeline = build_pipeline(config)
runner = PipelineRunner()
ctx = runner.run(pipeline, ctx)

result = ctx.state.get(Keys.RL_BACKTEST_RESULT)
if result:
    print(f"Pipeline backtest: n_episodes={result.n_episodes}, mean_return={result.mean_pnl_return:.4f}, sharpe={result.sharpe_ratio:.2f}")
else:
    print("No result (q_learning or pipeline step failed).")

## 4. Save and load an agent

To backtest a **trained** agent, save it with `save_agent`, then load it with `load_agent` (you must provide the same agent class/factory). Here we save our `RandomAgent` and reload it.

In [ ]:
from pathlib import Path
from src.q_learning.pipelines.inference import save_agent, load_agent

artifact_dir = Path("./artifacts/rl_tutorial_saved_agent")
artifact_dir.mkdir(parents=True, exist_ok=True)

save_agent(agent, str(artifact_dir), config={"n_actions": 3}, metadata={"tutorial": True})
print(f"Saved agent to {artifact_dir}")

loaded = load_agent(str(artifact_dir), agent_factory=RandomAgent, factory_kwargs={"n_actions": 3})
print("Loaded agent:", type(loaded).__name__)

# Quick backtest with loaded agent
result2 = BacktestRunner(agent=loaded, env=env, config=BacktestConfig(n_episodes=5)).run()
print(f"Loaded agent mean return: {result2.mean_pnl_return:.4f}")

## 5. Deploy pipeline (load agent into state)

Use **rl.deploy_agent** when the agent is stored under an artifact path and you want it in context state for a downstream pipeline (e.g. backtest or live). Set `rl.agent_path` and `rl.agent_factory` in params.

In [ ]:
from src.orchestrator.pipelines.rl.deploy_agent import build_pipeline as build_deploy_pipeline

config_deploy = RunConfig(
    pipeline="rl.deploy_agent",
    params={
        "rl": {
            "agent_path": "rl_tutorial_saved_agent",  # relative to artifacts root
            "agent_factory": RandomAgent,
            "agent_factory_kwargs": {"n_actions": 3},
        },
    },
)
ctx2 = Context(
    run_id="rl-deploy-1",
    cfg=config_deploy,
    logger=None,
    artifact_store=ArtifactStore(artifacts_root=Path("./artifacts")),
)
pipeline_deploy = build_deploy_pipeline(config_deploy)
ctx2 = PipelineRunner().run(pipeline_deploy, ctx2)
deployed_agent = ctx2.state.get(Keys.RL_AGENT)
print("Deployed agent in state:", deployed_agent is not None)

---
## Summary

- **Standalone backtest:** Build `BacktestRunner(agent, env, config)` and call `runner.run()`.
- **Pipeline backtest:** Use `rl.backtest_agent` with `params["rl"]["backtest"]`; optional `agent_path` + `agent_factory` for a saved agent.
- **Save/load:** `save_agent(agent, path)`, `load_agent(path, agent_factory, factory_kwargs)`.
- **Deploy pipeline:** `rl.deploy_agent` loads an agent into state for use by other pipelines or live runners.